# TTS Pipeline — Colab GPU Worker

This notebook runs the Abogen TTS engine + pipeline worker on Colab's free GPU.

## Before you start

**On your local machine**, expose Redis so Colab can reach it:
```bash
# Option A: ngrok (recommended)
ngrok tcp 6379
# → copy the  tcp://X.tcp.ngrok.io:XXXXX  URL

# Option B: cloudflared
cloudflared tunnel --url tcp://localhost:6379
```

Then fill in `REDIS_URL` and `WORKER_ID` in the Config cell below.

**MP3 output** goes to Google Drive → your local merger reads it from the same Drive mount.

In [ ]:
# ── 1. Mount Google Drive (MP3 output) ────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
OUTPUT_DIR = '/content/drive/MyDrive/tts-pipeline/outputs'
SPOOL_DIR  = '/content/spool'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(SPOOL_DIR,  exist_ok=True)
print('Drive mounted. Output dir:', OUTPUT_DIR)

In [ ]:
# ── TEMP: Push existing Drive output → server via rsync ──────────────────────
# Run this cell standalone to sync whatever is already in Google Drive to the server.
# Requires: Drive mounted (cell 1) + SSH secrets loaded (cell 2).
# Safe to re-run — rsync skips files already on the server.

import subprocess, os

DRIVE_OUTPUT = '/content/drive/MyDrive/tts-pipeline/outputs'  # adjust if needed

if not os.path.isdir(DRIVE_OUTPUT):
    raise RuntimeError(f'Drive path not found: {DRIVE_OUTPUT} — run cell 1 first.')
if not (SSH_HOST and SSH_USER and SSH_KEY):
    raise RuntimeError('SSH secrets not loaded — run cell 2 first.')

# Count what we're about to push
mp3_count = sum(1 for _, _, files in os.walk(DRIVE_OUTPUT) for f in files if f.endswith('.mp3'))
print(f'Found {mp3_count} MP3 files in {DRIVE_OUTPUT}')
print(f'Pushing to {SSH_USER}@{SSH_HOST}:{SSH_REMOTE_PATH} ...')

result = subprocess.run([
    'rsync', '-avz', '--progress',
    '-e', f'ssh -i {SSH_KEY_PATH} -p {SSH_PORT} -o StrictHostKeyChecking=no',
    DRIVE_OUTPUT + '/',          # trailing slash = contents only, not the dir itself
    f'{SSH_USER}@{SSH_HOST}:{SSH_REMOTE_PATH}/',
], capture_output=False, text=True)   # capture_output=False so progress prints live

if result.returncode != 0:
    print('rsync failed — check output above.')
else:
    print(f'\nDone. {mp3_count} files synced to server.')

In [ ]:
# ── DEBUG: Inspect raw SSH_KEY from Secrets ───────────────────────────────────
# Run this to see exactly what Colab gave us so we can fix the key format.
# Remove or skip this cell once SSH is working.

print(f'Length : {len(SSH_KEY)} chars')
print(f'Has \\\\n : {"\\\\n" in SSH_KEY}')
print(f'Has \\n  : {chr(10) in SSH_KEY}')
print(f'Has \\r  : {chr(13) in SSH_KEY}')
print(f'Lines   : {len(SSH_KEY.splitlines())}')
print()
print('--- First 60 chars (raw repr) ---')
print(repr(SSH_KEY[:60]))
print()
print('--- Last 60 chars (raw repr) ---')
print(repr(SSH_KEY[-60:]))

In [ ]:
# ── 2. Secrets ────────────────────────────────────────────────────────────────
# Add these in the Secrets panel (🔑 icon, left sidebar):
#
#   GITHUB_TOKEN      → ghp_...  (Personal Access Token — private repos only)
#   GITHUB_USER       → your GitHub username
#   NGROK_TOKEN       → ngrok authtoken (dashboard.ngrok.com → Your Authtoken)
#
#   SSH_HOST          → server IP or hostname  e.g. 10.0.0.1
#   SSH_USER          → SSH username           e.g. root
#   SSH_KEY           → private key contents   (paste the full -----BEGIN ... KEY-----)
#   SSH_REMOTE_PATH   → remote output dir      e.g. /opt/tts-node/outputs
#   SSH_PORT          → SSH port (optional, default 22)
#
from google.colab import userdata

GITHUB_TOKEN    = userdata.get('GITHUB_TOKEN')
GITHUB_USER     = userdata.get('GITHUB_USER')
NGROK_TOKEN     = userdata.get('NGROK_TOKEN')
SSH_HOST        = userdata.get('SSH_HOST')
SSH_USER        = userdata.get('SSH_USER')
SSH_KEY         = userdata.get('SSH_KEY')
SSH_REMOTE_PATH = userdata.get('SSH_REMOTE_PATH') or '/opt/tts-node/outputs'
SSH_PORT        = int(userdata.get('SSH_PORT') or 22)

if GITHUB_TOKEN and GITHUB_USER:
    import subprocess
    subprocess.run([
        'git', 'config', '--global', 'credential.helper',
        f'!f() {{ echo username={GITHUB_USER}; echo password={GITHUB_TOKEN}; }}; f'
    ], check=True)
    print(f'Git auth: {GITHUB_USER}')
else:
    print('Git auth: skipped (public repo)')

print(f'ngrok token:  {"ok" if NGROK_TOKEN     else "MISSING"}')
print(f'SSH host:     {"ok" if SSH_HOST        else "MISSING"}')
print(f'SSH key:      {"ok" if SSH_KEY         else "MISSING"}')

In [ ]:
# ── 3. Config — fill these in ─────────────────────────────────────────────────
REDIS_URL = 'redis://X.tcp.ngrok.io:XXXXX'   # ← your ngrok TCP URL
WORKER_ID = 'colab-gpu-1'                     # ← unique name for this node

# ABOGEN_HOST here is the URL the worker uses to call the local Abogen API.
# Note: Abogen's app.py uses ABOGEN_HOST as its bind address (0.0.0.0),
# so we keep that separate — see cell 6.
os.environ['REDIS_URL']       = REDIS_URL
os.environ['WORKER_ID']       = WORKER_ID
os.environ['ABOGEN_HOST']     = 'http://localhost:8808'
os.environ['OUTPUT_DIR']      = OUTPUT_DIR
os.environ['SPOOL_DIR']       = SPOOL_DIR
os.environ['USE_GPU']         = 'true'
os.environ['PROMETHEUS_PORT'] = '8000'
print('Config set.')

In [ ]:
# ── SSH setup + mount remote output dir ──────────────────────────────────────
import os, subprocess, textwrap

if not (SSH_HOST and SSH_USER and SSH_KEY):
    raise RuntimeError('SSH_HOST, SSH_USER and SSH_KEY must all be set in Secrets.')

# Colab Secrets can mangle multi-line values — reconstruct proper PEM format.
# Handles: literal \n sequences, spaces instead of newlines, collapsed headers.
SSH_KEY_PATH = '/root/.ssh/colab_key'
os.makedirs('/root/.ssh', exist_ok=True)

def _reconstruct_pem(raw: str) -> str:
    # Replace literal \n sequences first
    key = raw.replace('\\n', '\n').strip()
    # If still no internal newlines, the whole key is on one line — reformat it
    if '\n' not in key:
        # Split on the header/footer markers and reformat the base64 body
        import re
        m = re.match(r'(-----BEGIN [^-]+-----)([A-Za-z0-9+/=\s]+)(-----END [^-]+-----)', key)
        if m:
            header, body, footer = m.group(1), m.group(2).replace(' ', ''), m.group(3)
            # Wrap base64 body at 64 chars (standard PEM line length)
            wrapped = '\n'.join(body[i:i+64] for i in range(0, len(body), 64))
            key = f'{header}\n{wrapped}\n{footer}'
    return key + '\n'

with open(SSH_KEY_PATH, 'w') as f:
    f.write(_reconstruct_pem(SSH_KEY))
os.chmod(SSH_KEY_PATH, 0o600)

# Validate the key loaded correctly
check = subprocess.run(['ssh-keygen', '-y', '-f', SSH_KEY_PATH], capture_output=True, text=True)
if check.returncode != 0:
    raise RuntimeError(f'Private key is invalid or unsupported format:\n{check.stderr}\n'
                       'Ensure it is OpenSSH format (not PuTTY .ppk). '
                       'Convert with: ssh-keygen -p -m OpenSSH -f your_key')
print('SSH key loaded OK')

# Silence host-key prompts
with open('/root/.ssh/config', 'w') as f:
    f.write(textwrap.dedent(f"""\
        Host {SSH_HOST}
            StrictHostKeyChecking no
            UserKnownHostsFile /dev/null
            IdentityFile {SSH_KEY_PATH}
            Port {SSH_PORT}
    """))

# Test connection
result = subprocess.run(
    ['ssh', '-o', 'ConnectTimeout=10', f'{SSH_USER}@{SSH_HOST}',
     f'mkdir -p {SSH_REMOTE_PATH} && echo ok'],
    capture_output=True, text=True
)
if result.returncode != 0 or 'ok' not in result.stdout:
    raise RuntimeError(f'SSH connection failed:\n{result.stderr}')
print(f'SSH ok → {SSH_USER}@{SSH_HOST}:{SSH_REMOTE_PATH}')

# Mount remote output dir via sshfs
!apt-get install -y sshfs > /dev/null 2>&1
SSHFS_MOUNT = '/content/remote-outputs'
os.makedirs(SSHFS_MOUNT, exist_ok=True)
subprocess.run(['fusermount', '-uz', SSHFS_MOUNT], capture_output=True)

mount_result = subprocess.run([
    'sshfs',
    f'{SSH_USER}@{SSH_HOST}:{SSH_REMOTE_PATH}',
    SSHFS_MOUNT,
    '-o', f'IdentityFile={SSH_KEY_PATH},StrictHostKeyChecking=no,port={SSH_PORT}',
    '-o', 'reconnect,ServerAliveInterval=15,ServerAliveCountMax=3',
], capture_output=True, text=True)

if mount_result.returncode != 0:
    raise RuntimeError(f'sshfs mount failed:\n{mount_result.stderr}')

OUTPUT_DIR = SSHFS_MOUNT
os.environ['OUTPUT_DIR'] = OUTPUT_DIR
print(f'sshfs mounted → {SSHFS_MOUNT}')

In [ ]:
# ── SSH setup + mount remote output dir ──────────────────────────────────────
# Writes the private key to disk, tests the connection, then mounts the server's
# output directory via sshfs so the worker writes MP3s directly to the server.
# The local Google Drive mount is not needed when using SSH.
import os, subprocess, textwrap

if not (SSH_HOST and SSH_USER and SSH_KEY):
    raise RuntimeError('SSH_HOST, SSH_USER and SSH_KEY must all be set in Secrets.')

# Write private key
SSH_KEY_PATH = '/root/.ssh/colab_key'
os.makedirs('/root/.ssh', exist_ok=True)
with open(SSH_KEY_PATH, 'w') as f:
    # Normalize — Colab Secrets sometimes strips trailing newline
    f.write(SSH_KEY.strip() + '\n')
os.chmod(SSH_KEY_PATH, 0o600)

# Silence host-key prompts
with open('/root/.ssh/config', 'w') as f:
    f.write(textwrap.dedent(f"""\
        Host {SSH_HOST}
            StrictHostKeyChecking no
            UserKnownHostsFile /dev/null
            IdentityFile {SSH_KEY_PATH}
            Port {SSH_PORT}
    """))

# Test connection
result = subprocess.run(
    ['ssh', '-o', 'ConnectTimeout=10', f'{SSH_USER}@{SSH_HOST}',
     f'mkdir -p {SSH_REMOTE_PATH} && echo ok'],
    capture_output=True, text=True
)
if result.returncode != 0 or 'ok' not in result.stdout:
    raise RuntimeError(f'SSH connection failed:\n{result.stderr}')
print(f'SSH ok → {SSH_USER}@{SSH_HOST}:{SSH_REMOTE_PATH}')

# Mount remote output dir via sshfs
!apt-get install -y sshfs > /dev/null 2>&1
SSHFS_MOUNT = '/content/remote-outputs'
os.makedirs(SSHFS_MOUNT, exist_ok=True)

# Unmount if previously mounted (re-run safe)
subprocess.run(['fusermount', '-uz', SSHFS_MOUNT], capture_output=True)

mount_result = subprocess.run([
    'sshfs',
    f'{SSH_USER}@{SSH_HOST}:{SSH_REMOTE_PATH}',
    SSHFS_MOUNT,
    '-o', f'IdentityFile={SSH_KEY_PATH},StrictHostKeyChecking=no,port={SSH_PORT}',
    '-o', 'reconnect,ServerAliveInterval=15,ServerAliveCountMax=3',
], capture_output=True, text=True)

if mount_result.returncode != 0:
    raise RuntimeError(f'sshfs mount failed:\n{mount_result.stderr}')

# Override OUTPUT_DIR to point at the sshfs mount
OUTPUT_DIR = SSHFS_MOUNT
os.environ['OUTPUT_DIR'] = OUTPUT_DIR
print(f'sshfs mounted → {SSHFS_MOUNT} (writes go directly to server)')

In [ ]:
# ── Post-install GPU check ────────────────────────────────────────────────────
import torch
if not torch.cuda.is_available():
    raise RuntimeError('PyTorch cannot see the GPU — CUDA install may have failed. Check cell 5 output.')
print(f'PyTorch {torch.__version__} | CUDA {torch.version.cuda} | Device: {torch.cuda.get_device_name(0)}')

In [ ]:
# ── 4. Install system deps ────────────────────────────────────────────────────
!apt-get install -y espeak-ng ffmpeg > /dev/null 2>&1
print('espeak-ng + ffmpeg installed.')

In [ ]:
# ── 5. Install Abogen + GPU PyTorch ──────────────────────────────────────────
!pip install -q torch torchaudio --index-url https://download.pytorch.org/whl/cu121
!git clone --depth=1 https://github.com/wizardgang/audiobook-stack /content/audiobook-stack 2>/dev/null || \
 git -C /content/audiobook-stack pull
!pip install -q -e '/content/audiobook-stack/abogen_src[gpu-cu121]' 2>/dev/null || \
 pip install -q -e '/content/audiobook-stack/abogen_src'
!pip install -q redis prometheus_client requests pyngrok
print('All packages installed.')

In [ ]:
# ── 6. Start Abogen TTS server in background ─────────────────────────────────
# Abogen's app.py reads ABOGEN_HOST as its bind address (default 0.0.0.0)
# and ABOGEN_PORT for the port. We pass these explicitly so the worker's
# ABOGEN_HOST=http://localhost:8808 in os.environ doesn't bleed through.
import subprocess, time

abogen_env = {**os.environ, 'ABOGEN_HOST': '0.0.0.0', 'ABOGEN_PORT': '8808'}
abogen_proc = subprocess.Popen(
    ['python', '-m', 'abogen.webui.app'],
    cwd='/content/audiobook-stack',
    env=abogen_env,
    stdout=open('/content/abogen.log', 'w'),
    stderr=subprocess.STDOUT,
)

import requests as _req
for i in range(30):
    try:
        if _req.get('http://localhost:8808/api/voices', timeout=2).ok:
            print(f'Abogen ready after {i*2}s')
            break
    except:
        pass
    time.sleep(2)
else:
    print('Abogen did not start — check /content/abogen.log')
    !tail -20 /content/abogen.log

In [ ]:
# ── 7. Verify Redis connection ────────────────────────────────────────────────
import redis
r = redis.from_url(REDIS_URL, decode_responses=True, socket_connect_timeout=5)
print('Redis ping:', r.ping())
print('TTS queue depth:', r.llen('pipeline:tts'))

In [ ]:
# ── 8. Start worker ───────────────────────────────────────────────────────────
import threading, importlib.util
import prometheus_client
import prometheus_client.registry as _prom_reg

# Replace the global registry with a fresh one so re-running this cell
# never raises "Duplicated timeseries in CollectorRegistry".
_fresh = prometheus_client.CollectorRegistry(auto_describe=True)
prometheus_client.REGISTRY = _fresh
_prom_reg.REGISTRY = _fresh

spec = importlib.util.spec_from_file_location('worker', '/content/audiobook-stack/worker/worker.py')
worker_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(worker_mod)

t = threading.Thread(target=worker_mod.main, daemon=True, name='pipeline-worker')
t.start()
print(f'Worker {WORKER_ID} started — consuming pipeline:tts')
print(f'Abogen: http://localhost:8808 | Redis: {REDIS_URL}')

In [ ]:
# ── 9. Expose Prometheus metrics via ngrok ────────────────────────────────────
# Opens an HTTP tunnel to port 8000 so your local Prometheus can scrape this worker.
# After running this cell, add the printed URL to config/prometheus.yml:
#
#   - job_name: 'workers'
#     static_configs:
#       - targets:
#         - 'host.docker.internal:8000'  # local worker
#         - '<ngrok-host>:<ngrok-port>'  # ← paste the host:port printed below
#
from pyngrok import ngrok, conf

if not NGROK_TOKEN:
    print('Skipping — no NGROK_TOKEN set in Secrets.')
else:
    conf.get_default().auth_token = NGROK_TOKEN
    tunnel = ngrok.connect(8000, 'http')
    public_url = tunnel.public_url

    # Extract host:port for prometheus.yml (strip https://)
    from urllib.parse import urlparse
    parsed = urlparse(public_url)
    prom_target = f'{parsed.hostname}:443'

    print(f'Prometheus tunnel: {public_url}')
    print(f'\nAdd to config/prometheus.yml → targets:')
    print(f"  - '{prom_target}'")

In [ ]:
# ── 9. Keep-alive + live status (run this cell last) ─────────────────────────
import time

r2 = redis.from_url(REDIS_URL, decode_responses=True)
while True:
    try:
        queue_depth = r2.llen('pipeline:tts')
        heartbeat   = r2.hget('worker:heartbeats', WORKER_ID) or '0'
        age = int(time.time()) - int(heartbeat)
        print(f'\r[{time.strftime("%H:%M:%S")}] queue={queue_depth} | {WORKER_ID} last_seen={age}s ago   ', end='')
    except Exception as e:
        print(f'\nRedis error: {e}')
    time.sleep(10)